<a href="https://colab.research.google.com/github/mattany/beyond-accuracy/blob/main/training/sft/GPT_SFT_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="text-align:center; font-size:40px;">Fine Tuning Llama 2</h1>
<h2 style="text-align:center; font-size:20px;">Teaching LLMs to Generate Better Scientific Explanations</h2>
<p style="text-align: center">
  <img src="https://github.com/OdedReg/Fine-Tuning/blob/main/Llama.jpg?raw=true" width="600px">
</p>


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
import os
os.environ['HF_HOME'] = '/content/beyond-accuracy/.cache/huggingface'


In [ ]:
import os
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])


In [ ]:
MODEL_SAVE_PATH = "/content/beyond-accuracy/models/Llama-2-7b-chat-hf-science-sft/"
SFT_DS_PATH = "/content/beyond-accuracy/data/qa_pairs/ask_science_gpt_answers.csv"

In [ ]:
# Legacy hard-coded token removed; use HF_TOKEN from the environment.


In [ ]:
%pip install datasets
%pip install peft
%pip install trl
%pip install bitsandbytes

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.4/318.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 6.6 MB/s eta 0:00:00


In [ ]:
# import neptune
import json
import re
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from trl import SFTTrainer, RewardTrainer, PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead, create_reference_model
from tqdm import tqdm
import sklearn
from sklearn.model_selection import train_test_split
import zipfile
import os
from getpass import getpass
from tqdm import tqdm
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns
import time
import random
import warnings

# Suppress FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Data

In [ ]:
def formatting_prompts_func(example):
    """
    Formats a list of questions and corresponding answers into a list of formatted text strings.

    Parameters:
        example (dict): A dictionary containing lists of questions and answers.

    Returns:
        list: A list of formatted text strings where each string contains a question and its corresponding answer.
    """
    output_texts = []
    for i in range(len(example['Question'])):
        text = f"[INST] {example['Question'][i]} [/INST] {example['Answer'][i]} </s>"

        output_texts.append(text)
    return output_texts

In [ ]:
df = pd.read_csv(SFT_DS_PATH)
df["Dataset"] = "ask_science"
df = df[df["Truncated"] == 0]
print(df.count())
df.head()

Index        9083
Question     9083
Answer       9083
Truncated    9083
Dataset      9083
dtype: int64


,Index,Question,Answer,Truncated,Dataset
0,0,how do centipedes/millipedes control all of th...,Centipedes and millipedes have multiple pairs ...,0,ask_science
1,1,what is the significance of the derivative of ...,Understanding the significance of the derivati...,0,ask_science
2,2,do we live in a cause-and-effect universe? are...,"In our world, the principle of cause-and-effec...",0,ask_science
3,3,how closely related any two people on average?,"On average, any two people on Earth are estima...",0,ask_science
4,4,do primates ever keep the tools they fashion? ...,"When it comes to primates and their tool use, ...",0,ask_science


## Filter and Process Dataset

### Filter Dataset


In [ ]:
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
# MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, device_map="auto")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

df['text'] = df.apply(lambda row: f"[INST] {row['Question']} [/INST] {row['Answer']} </s>", axis=1)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def print_avg_length(df, col_name, text):
    """
    Prints the average length of the specified column in the DataFrame.

    Args:
    df (DataFrame): DataFrame containing the data.
    col_name (str): Name of the column for which average length is to be calculated.
    text (str): Text to display along with the average length.

    Returns:
    None
    """
    col_lengths = df[col_name].apply(len)
    col_avg_lengths= col_lengths.mean()
    print(text, int(col_avg_lengths))


### Process Dataset

#### Add token lists for each field

In [ ]:
df['Question_list'] = df['Question'].apply(tokenizer.encode)
df['Answer_list'] = df['Answer'].apply(tokenizer.encode)
df['text_list'] = df['text'].apply(tokenizer.encode)


#### Split dataset

In [ ]:
train_df, eval_df = train_test_split(df, test_size=0.02, random_state=42)
# eval_df, test_df = train_test_split(eval_df, test_size=0.5, random_state=42)

In [ ]:
eval_df.head()

,Index,Question,Answer,Truncated,Dataset,text
7383,7383,"what are the ""bumps"" or ""spikes"" on a dogs lips?","The ""bumps"" or ""spikes"" you see on a dog's lip...",0,ask_science,"[INST] what are the ""bumps"" or ""spikes"" on a d..."
4021,4021,what's actually happening when you go temporar...,"When you stand up too quickly, especially from...",0,ask_science,[INST] what's actually happening when you go t...
2360,2360,will a tesla eventually not hold its charge as...,"Over time, all rechargeable batteries, includi...",0,ask_science,[INST] will a tesla eventually not hold its ch...
3196,3196,how does sound travel through items? like wall...,Sound travels through items like walls or wind...,0,ask_science,[INST] how does sound travel through items? li...
8705,8705,does tire pressure change when they are in mot...,"When a tire is in motion (rotating), the tire ...",0,ask_science,[INST] does tire pressure change when they are...


In [ ]:
df.head()

,Index,Question,Answer,Truncated,Dataset,text
0,0,how do centipedes/millipedes control all of th...,Centipedes and millipedes have multiple pairs ...,0,ask_science,[INST] how do centipedes/millipedes control al...
1,1,what is the significance of the derivative of ...,Understanding the significance of the derivati...,0,ask_science,[INST] what is the significance of the derivat...
2,2,do we live in a cause-and-effect universe? are...,"In our world, the principle of cause-and-effec...",0,ask_science,[INST] do we live in a cause-and-effect univer...
3,3,how closely related any two people on average?,"On average, any two people on Earth are estima...",0,ask_science,[INST] how closely related any two people on a...
4,4,do primates ever keep the tools they fashion? ...,"When it comes to primates and their tool use, ...",0,ask_science,[INST] do primates ever keep the tools they fa...


## .

### Remove answers that are too long or too short


In [ ]:
# Sort the DataFrame by text_length
df['text_length'] = df['text_list'].apply(lambda x: len(x))
df = df.sort_values(by='text_length')

# Remove short answers (less than 70 tokens)
df = df[(256 >= df['text_length']) & (df['text_length'] >= 150)]

df.reset_index(drop=True, inplace=True)

### Calculate the cumulative count of rows until each text length

In [ ]:

df['cumulative_count'] = range(1, len(df) + 1)

# Plotting the cumulative graph
plt.plot(df['text_length'], df['cumulative_count'], marker='o')
plt.xlabel('Text Length')
plt.ylabel('Cumulative Count of Rows')
plt.title('Cumulative Count of Rows Until Text Length')
plt.grid(True)
plt.show()

### EDA

In [ ]:
num_rows = df.shape[0]
print("Number of Q/A in the df:", num_rows)

In [ ]:
print("Number of questions and answers from each dataset:\n")
print(df['Dataset'].value_counts())

In [ ]:
# Plotting histogram of Datasets
plt.figure(figsize=(5, 4))
df['Dataset'].value_counts().plot(kind='bar', color='brown')
plt.title('Histogram of Dataset')
plt.xlabel('Dataset')
plt.ylabel('Number of Q/A')
plt.xticks(rotation=45)
plt.show()

In [ ]:

random.seed(42)
indexes = [_ for _ in range(len(df))]

# Get 10 random elements from the list
sampled_items = random.sample(indexes, 10)

for i in range(10):
    dataset_name = df.iloc[i]['Dataset']
    question = df.iloc[i]['Question']
    answer = df.iloc[i]['Answer']
    print(f'Dataset name: {dataset_name}\n')
    print(f'Question:\n{question}\n')
    print(f'Answer:\n{answer}')
    print("\n\n------------------------------------------")

In [ ]:
train_df.iloc[0]['Question']

In [ ]:
print("Data stats:")
print_avg_length(df, 'Question_list', "Average question length:")
print_avg_length(df, 'Answer_list', "Average answer length:")
print_avg_length(df, 'text_list', "Average text length:")
print("\n----------------------------------")

print("Train stats:")
print_avg_length(train_df, 'Question_list', "Average question length:")
print_avg_length(train_df, 'Answer_list', "Average answer length:")
print_avg_length(train_df, 'text_list', "Average text length:")
print("\n----------------------------------")

print("eval stats:")
print_avg_length(eval_df, 'Question_list', "Average question length:")
print_avg_length(eval_df, 'Answer_list', "Average answer length:")
print_avg_length(eval_df, 'text_list', "Average text length:")
print("\n----------------------------------")

print("test stats:")
print_avg_length(test_df, 'Question_list', "Average question length:")
print_avg_length(test_df, 'Answer_list', "Average answer length:")
print_avg_length(test_df, 'text_list', "Average text length:")


### Convert to Huggingface Datasets format

#### We will use a token limit of 256 for both tokenization and generation for several reasons:
1. Conciseness: Concise explanations are more effective than lengthy ones as they distill complex concepts and prioritize key details.
2. Efficiency: This choice helps streamline processing, reducing overall running time.
3. User-Friendly: It simplifies tasks for human annotators, making their work more manageable.
4. Balance: With 256 tokens, we want to ensure effectiveness without unnecessary verbosity.

In [ ]:
train_df.reset_index(drop=True, inplace=True)
eval_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)



train = Dataset.from_pandas(train_df)
eval = Dataset.from_pandas(eval_df)

In [ ]:
# Define the columns to select

# select the text column(s)
train = train.select_columns(['text'])

# select the text column(s)
eval = eval.select_columns(['text'])

In [ ]:
train

In [ ]:
eval

## SFT Model

In [ ]:
def create_model_and_tokenizer(MODEL_NAME):
    """
    Creates a language model and tokenizer from the specified model name.

    Parameters:
        MODEL_NAME (str): The path of the pre-trained model.

    Returns:
        tuple: A tuple containing the language model and tokenizer.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        use_safetensors=True,
        quantization_config=bnb_config,
        trust_remote_code=True,
        device_map="auto",
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, device_map="auto", trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer

In [ ]:
def print_number_of_trainable_model_parameters(model):
    """
    Calculate and print the number of trainable and total parameters of a PyTorch model.

    Parameters:
    - model (torch.nn.Module): The PyTorch model whose parameters are to be counted.

    Returns:
    - str: A formatted string containing the number of trainable and total parameters,
           along with the percentage of trainable parameters.

    """
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()

    # Add commas to the numbers using format()
    trainable_model_params_str  = "{:,}".format(trainable_model_params)
    all_model_params_str  = "{:,}".format(all_model_params)

    return f"Trainable model parameters: {trainable_model_params_str}\nAll model parameters: {all_model_params_str}\nPercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

def clean_text(text):
    """
    Cleans the input text by removing the last part if it contains the word 'tokens' or 'token'.

    This function checks if the input text contains any newline characters. If it does,
    it splits the text into parts using the newline character as a delimiter. It then
    examines the last part of the split text. If the last part contains the word 'tokens'
    or 'token' (case insensitive), it removes this part and returns the rest of the text.
    Otherwise, it returns the original text.

    Parameters:
    text (str): The input text to be cleaned.

    Returns:
    str: The cleaned text with the last part removed if it contains 'tokens' or 'token',
         otherwise the original text.
    """
    if '\n' in text:
        parts = text.split('\n')
        last_part = parts[-1]

        if ('tokens' in last_part.lower()) or ('token' in last_part.lower()):
            return '\n'.join(parts[:-1])  # Return the text without the last part
    return text

In [ ]:
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
model, tokenizer = create_model_and_tokenizer(MODEL_NAME)
model.config.use_cache = False

In [ ]:
print(print_number_of_trainable_model_parameters(model))

When applying quantization to a neural network, some weights and activations may become redundant and can be removed.

This explains why we got 3.5B parameters instead of 7B

In [ ]:
assert False

## Training

In [ ]:
lora_r = 16
lora_alpha = 64
lora_dropout = 0.1
lora_target_modules = [
    "q_proj",
    "up_proj",
    "o_proj",
    "k_proj",
    "down_proj",
    "gate_proj",
    "v_proj",
]


peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=lora_target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

training_arguments = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-5,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    num_train_epochs=2,
    eval_steps=20,
    save_steps=20,
    logging_steps=20,
    warmup_ratio=0.05,
    evaluation_strategy="steps",
    save_strategy="steps",
    group_by_length=True,
    # output_dir="Llama-2-7b-chat-hf-science-sft", # commented so I don't overwrite by mistake
    load_best_model_at_end=True,
    # report_to="neptune",
    save_safetensors=True,
    lr_scheduler_type="cosine",
    seed=42,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train,
    eval_dataset=eval,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=256,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=False,
)

In [ ]:
print(print_number_of_trainable_model_parameters(model))

Peft reduced the precentage of trainable parameters from 7.5% to 1.13%

In [ ]:
trainer.train()

### Run


In [ ]:
trainer.save_model(MODEL_SAVE_PATH)

In [ ]:
# Reload model in FP16 and merge it with LoRA weights
base_model = "meta-llama/Llama-2-7b-chat-hf"
new_model = MODEL_SAVE_PATH
load_model = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(load_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b-chat-hf.
401 Client Error. (Request ID: Root=1-66eda37d-2df4491722875be15775c91c;69049274-254c-4bb6-b9f2-f086e70c43db)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-chat-hf/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b-chat-hf is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
model.push_to_hub("Llama-2-7b-chat-hf-science-sft", use_temp_dir=False)
tokenizer.push_to_hub("Llama-2-7b-chat-hf-science-sft", use_temp_dir=False)